<h1 style="text-align: center;">Programa 9 — Validación cruzada 2</h1>
<p style="text-align: center;">20-11-2026</p>
<p style="text-align: center;">
    <img src="img/ITQ.png" width="500">
</p>

**Nombre:** Nikolay Misael De La Cruz Melendres  
**Asignatura:** Fundamentos de Machine Learning  
**Docente:** Elvis David Pachacama Cabezas  
**Repositorio:** [NikolayDlc/fundamentosMachine](https://github.com/NikolayDlc/fundamentosMachine)


<div align="center">
---

**Propósito:** evaluar un clasificador de flores Iris mediante validación cruzada, reservar un conjunto de test externo, entrenar el modelo definitivo y predecir una flor nueva. El desarrollo conserva el enfoque del programa de clase y separa las explicaciones del código.


## 1. Importación de bibliotecas

- `numpy`: operaciones numéricas y matrices.
- `datasets`: obtiene Iris, un conjunto incluido en scikit-learn.
- `train_test_split`: divide observaciones en entrenamiento y test.
- `KFold` y `cross_val_score`: forman cinco particiones y evalúan un modelo en cada una.
- `StandardScaler`: estandariza las medidas de las flores.
- `SVC`: clasifica las especies.
- `make_pipeline`: encadena escalador y clasificador **durante** la validación cruzada, de modo que cada fold utiliza solo su entrenamiento para ajustar la escala.


In [1]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score

## 2. Carga del dataset Iris

Iris contiene **150 observaciones**, **4 características** por flor y **3 clases**: setosa, versicolor y virginica. Cada fila representa una flor y cada columna representa una medida. Las etiquetas numéricas 0, 1 y 2 corresponden al orden de `target_names`.

In [2]:
iris = datasets.load_iris()
X = iris.data
y = iris.target

print("Especies:", iris.target_names)
print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

valores, cantidades = np.unique(y, return_counts=True)
print("Etiquetas:", valores)
print("Cantidad por clase:", cantidades)

Especies: ['setosa' 'versicolor' 'virginica']
Dimensiones de X: (150, 4)
Dimensiones de y: (150,)
Etiquetas: [0 1 2]
Cantidad por clase: [50 50 50]


## 3. Hold-Out externo: 80 % para entrenamiento y 20 % para test

`test_size=0.2` separa **120 flores de entrenamiento** y **30 de test**. `random_state=42` fija la división aleatoria para poder reproducirla. **Las 30 flores de test quedan fuera de KFold** y se usan al evaluar el modelo definitivo.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Test:", X_test.shape)

valores_test, cantidades_test = np.unique(y_test, return_counts=True)
print("Clases del test:", valores_test)
print("Cantidad por clase en test:", cantidades_test)

Entrenamiento: (120, 4)
Test: (30, 4)
Clases del test: [0 1 2]
Cantidad por clase en test: [10  9 11]


## 4. KFold interno: cinco evaluaciones

Con 120 observaciones y cinco folds, **cada repetición entrena con 96 flores y valida con 24**. `cross_val_score` crea y ajusta un modelo en cada fold: no evalúa cinco veces un único modelo ya entrenado.

Se mantiene `KFold(n_splits=5)` y `SVC(C=0.5, gamma="auto")` como en el programa original. La diferencia técnica es que aquí la estandarización ocurre **dentro de cada fold**, evitando que los datos de validación influyan en las medias y desviaciones del escalador. `KFold` sin `shuffle=True` conserva el orden recibido.

**Importante:** este ejemplo evalúa un valor fijo de `C=0.5`; no realiza una búsqueda automática del mejor hiperparámetro.

In [4]:
modelo_validacion = make_pipeline(
    StandardScaler(),
    SVC(C=0.5, gamma="auto")
)

kf = KFold(n_splits=5)

resultados = cross_val_score(
    modelo_validacion,
    X_train,
    y_train,
    cv=kf,
    scoring="accuracy"
)

print("Exactitudes por fold:", resultados)
print("Accuracy media: %.4f" % resultados.mean())
print("Desviación estándar: %.4f" % resultados.std())
print("Accuracy media en porcentaje: %.2f %%" % (resultados.mean() * 100))

Exactitudes por fold: [0.95833333 0.95833333 0.83333333 0.95833333 0.95833333]
Accuracy media: 0.9333
Desviación estándar: 0.0500
Accuracy media en porcentaje: 93.33 %


### Cómo leer los resultados

- Cada posición de `resultados` es la proporción de aciertos en **un fold**.
- `resultados.mean()` resume la exactitud promedio entre las cinco evaluaciones.
- `resultados.std()` muestra cuánto varían las exactitudes de un fold a otro; **no es la tasa de error del modelo**.
- Estos resultados **no son la evaluación final del test externo**.

## 5. Estandarización para el modelo definitivo

Ahora se ajusta **un nuevo escalador** con las 120 flores del entrenamiento externo. Se aplica ese mismo escalador a test mediante `transform()` y no mediante `fit_transform()`. Esto reproduce la lógica de preparación del programa de clase sin introducir datos del test en el ajuste de escala.

La matriz `X_train` original se conserva para que el proceso sea fácil de seguir.

In [5]:
standardizer = StandardScaler()

X_train_estandarizado = standardizer.fit_transform(X_train)
X_test_estandarizado = standardizer.transform(X_test)

print("Entrenamiento estandarizado:", X_train_estandarizado.shape)
print("Test estandarizado:", X_test_estandarizado.shape)

Entrenamiento estandarizado: (120, 4)
Test estandarizado: (30, 4)


## 6. Entrenamiento del clasificador definitivo

En validación cruzada se ajustaron cinco modelos temporales. Aquí se entrena **otro SVC** utilizando todas las 120 flores de entrenamiento externo. El conjunto de test todavía no se utiliza para ajustar el modelo.

In [6]:
svc = SVC(C=0.5, gamma="auto")
svc.fit(X_train_estandarizado, y_train)

print("Modelo definitivo entrenado.")

Modelo definitivo entrenado.


## 7. Exactitud sobre entrenamiento

`score()` calcula la proporción de etiquetas correctas del conjunto indicado. La exactitud de entrenamiento permite observar cómo se comporta el modelo con datos que sí vio durante `fit()`. **No sustituye la evaluación externa**.

In [7]:
score_train = svc.score(X_train_estandarizado, y_train)

print("Exactitud en entrenamiento: %.2f %%" % (score_train * 100))

Exactitud en entrenamiento: 95.83 %


## 8. Evaluación final sobre test externo

Se utilizan las 30 flores separadas al principio. Esta exactitud mide cómo clasifica el modelo definitivo las observaciones reservadas. No conviene seleccionar valores de `C` mirando repetidamente este resultado.

In [8]:
score_test = svc.score(X_test_estandarizado, y_test)

print("Exactitud en test: %.2f %%" % (score_test * 100))

Exactitud en test: 100.00 %


## 9. Predicciones y comparación con las etiquetas reales

`predict()` recibe las características de test y devuelve una etiqueta para cada flor. `y_test` contiene las respuestas correctas. Al compararlas, se puede identificar qué casos fueron clasificados correctamente.

In [9]:
y_pred = svc.predict(X_test_estandarizado)

print("Predicciones:    ", y_pred)
print("Etiquetas reales:", y_test)
print("Aciertos:", np.sum(y_pred == y_test))
print("Total en test:", len(y_test))

Predicciones:     [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Etiquetas reales: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Aciertos: 30
Total en test: 30


## 10. Matriz de confusión (ampliación)

Esta celda complementa el programa original con una matriz de confusión. En Iris hay tres clases: **filas = clases reales**, **columnas = clases predichas**. La diagonal principal contiene los aciertos. Los valores fuera de esa diagonal indican confusiones entre especies.

In [10]:
matriz = confusion_matrix(y_test, y_pred)
exactitud = accuracy_score(y_test, y_pred)

print("Orden de las especies:", iris.target_names)
print("Matriz de confusión:")
print(matriz)
print("Accuracy:", round(exactitud * 100, 2), "%")

Orden de las especies: ['setosa' 'versicolor' 'virginica']
Matriz de confusión:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
Accuracy: 100.0 %


## 11. Predicción de una flor nueva: ejemplo listo para ejecutar

Las cuatro medidas se introducen **en el mismo orden** que utiliza Iris: largo del sépalo, ancho del sépalo, largo del pétalo y ancho del pétalo. Una sola flor necesita una matriz con forma `(1, 4)`. `standardizer.transform()` aplica la escala aprendida durante entrenamiento. `svc.predict()` entrega una etiqueta entre 0 y 2, que se convierte en nombre utilizando `iris.target_names`.

In [11]:
entrada = np.array([[5.1, 3.5, 1.4, 0.2]])

entrada_estandarizada = standardizer.transform(entrada)
prediccion = svc.predict(entrada_estandarizada)
clase = int(prediccion[0])

print("Forma de entrada:", entrada.shape)
print("Clase predicha:", clase)
print("Especie predicha:", iris.target_names[clase])

Forma de entrada: (1, 4)
Clase predicha: 0
Especie predicha: setosa


## 12. Predicción por teclado (opcional)

Esta celda permite escribir tus propias cuatro medidas. Al ejecutarla, Jupyter pedirá cuatro números. Debes ejecutar primero todas las celdas anteriores, porque utiliza `standardizer` y `svc` ya ajustados.

In [ ]:
largo_sepalo = float(input("Largo del sépalo (cm): "))
ancho_sepalo = float(input("Ancho del sépalo (cm): "))
largo_petalo = float(input("Largo del pétalo (cm): "))
ancho_petalo = float(input("Ancho del pétalo (cm): "))

entrada_manual = np.array([[
    largo_sepalo,
    ancho_sepalo,
    largo_petalo,
    ancho_petalo
]])

entrada_manual_estandarizada = standardizer.transform(entrada_manual)
prediccion_manual = svc.predict(entrada_manual_estandarizada)
clase_manual = int(prediccion_manual[0])

print("Clase predicha:", clase_manual)
print("Especie predicha:", iris.target_names[clase_manual])